# 25. Project Three: An RNA-seq Study from Counts to Biology

The programs of this chapter, in the order the book prints
them. Each cell holds the file of the same name in this folder,
so editing a cell does not change the script beside it.

Run the cells from this folder, where the data files the chapter
uses live. Some cells call command line tools or download data,
and a few take minutes.


## `download.py`

Download the experiment and the gene sets it will be interpreted with.


In [ ]:
"""Download the experiment and the gene sets it will be interpreted with.

The counts and sample descriptions are the GSE52778 files of Chapter 18,
and gene_info gives each NCBI GeneID its symbol.
The gene sets are the Gene Ontology: goa_human.gaf.gz says which genes are
annotated to which terms, and go-basic.obo says what each term means and
which terms contain which.

uv run download.py          writes five files, about 50 MB
"""

import os
import urllib.request

GEO = "https://www.ncbi.nlm.nih.gov/geo/download/"
SERIES = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE52nnn/GSE52778"
GENES = ("https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia"
         "/Homo_sapiens.gene_info.gz")
GOA = "https://ftp.ebi.ac.uk/pub/databases/GO/goa/HUMAN/goa_human.gaf.gz"
OBO = "https://current.geneontology.org/ontology/go-basic.obo"

FILES = [
    (f"{GEO}?type=rnaseq_counts&acc=GSE52778&format=file"
     "&file=GSE52778_raw_counts_GRCh38.p13_NCBI.tsv.gz", "counts.tsv.gz"),
    (f"{SERIES}/matrix/GSE52778_series_matrix.txt.gz",
     "series_matrix.txt.gz"),
    (GENES, "gene_info.tsv.gz"),
    (GOA, "goa_human.gaf.gz"),
    (OBO, "go-basic.obo"),
]


def fetch(url, name):
    headers = {"User-Agent": "python"}
    request = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(request, timeout=900) as reply:
        with open(name + ".part", "wb") as out:
            while chunk := reply.read(1 << 16):
                out.write(chunk)
    os.replace(name + ".part", name)
    print(f"{name}, {os.path.getsize(name) / 1e6:.1f} MB")


if __name__ == "__main__":
    for url, name in FILES:
        fetch(url, name)

## `de.py`

Which genes dexamethasone changes, with each donor as its own control.


In [ ]:
"""Which genes dexamethasone changes, with each donor as its own control.

Chapter 19's program, reading this folder's files and writing the full
table of results for the steps that follow.

uv run de.py                writes results.tsv
"""

import gzip

import pandas as pd
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

KEEP = ["Untreated", "Dexamethasone"]


def symbols():
    """{GeneID: symbol} from NCBI's gene_info."""
    names = {}
    with gzip.open("gene_info.tsv.gz", "rt") as handle:
        header = handle.readline().lstrip("#").rstrip("\n").split("\t")
        for line in handle:
            row = dict(zip(header, line.rstrip("\n").split("\t")))
            names[int(row["GeneID"])] = row["Symbol"]
    return names


def main():
    counts = pd.read_csv("counts.tsv.gz", sep="\t", index_col="GeneID")
    samples = pd.read_csv("samples.tsv", sep="\t", index_col="sample")
    samples = samples[samples["treatment"].isin(KEEP)]
    counts = counts[samples.index].T
    counts = counts.loc[:, counts.sum() >= 10]

    inference = DefaultInference(n_cpus=2)
    dataset = DeseqDataSet(counts=counts, metadata=samples,
                           design="~cell_line + treatment",
                           inference=inference, quiet=True)
    dataset.deseq2()
    stats = DeseqStats(dataset,
                       contrast=["treatment", "Dexamethasone",
                                 "Untreated"],
                       inference=inference, quiet=True)
    stats.summary()
    results = stats.results_df
    results.index = results.index.astype(int)
    results.insert(0, "symbol",
                   pd.Series(symbols()).reindex(results.index))
    results.to_csv("results.tsv", sep="\t")

    significant = results["padj"] < 0.05
    changed = significant & (results["log2FoldChange"].abs() >= 1)
    print(f"{len(results):,} genes tested, "
          f"{significant.sum():,} with adjusted p below 0.05")
    print(f"{changed.sum():,} of those changed at least twofold")


if __name__ == "__main__":
    main()

## `enrich.py`

Name the biology behind a list of changed genes.


In [ ]:
"""Name the biology behind a list of changed genes.

A list of 945 gene symbols is not an answer. Enrichment asks, for each
set of genes that share a known role, whether more of that set changed
than chance would give. The sets here are Gene Ontology biological
process terms, and the test is the hypergeometric one: drawing 492 genes
from 14,053, how surprising is it to draw 7 of the 34 that respond to
glucocorticoids?

uv run enrich.py            writes enrichment.tsv
"""

import csv
import gzip

import pandas as pd
from scipy.stats import hypergeom

PROCESS = "biological_process"
SMALLEST, LARGEST = 10, 500  # terms outside this range are not testable
MIN_FOUND = 3  # a term needs at least this many changed genes to report


def ontology(path="go-basic.obo"):
    """{term: (name, namespace)} and {term: parents}, from the OBO."""
    names, parents, current = {}, {}, None
    with open(path) as handle:
        for line in handle:
            line = line.rstrip("\n")
            if line == "[Term]":
                current = {"parents": set()}
            elif line == "" and current is not None:
                if current.get("id") and not current.get("obsolete"):
                    names[current["id"]] = (current.get("name", ""),
                                            current.get("namespace", ""))
                    parents[current["id"]] = current["parents"]
                current = None
            elif current is not None and ": " in line:
                key, value = line.split(": ", 1)
                if key in ("id", "name", "namespace"):
                    current[key] = value
                elif key == "is_a":
                    current["parents"].add(value.split(" ! ")[0])
                elif key == "relationship":
                    kind, parent = value.split()[:2]
                    if kind == "part_of":
                        current["parents"].add(parent)
                elif key == "is_obsolete":
                    current["obsolete"] = True
    return names, parents


def annotations(path="goa_human.gaf.gz"):
    """{gene symbol: terms it is directly annotated to}."""
    direct = {}
    with gzip.open(path, "rt") as handle:
        for line in handle:
            if line.startswith("!"):
                continue
            column = line.split("\t")
            if "NOT" in column[3]:  # an annotation saying it does not
                continue
            direct.setdefault(column[2], set()).add(column[4])
    return direct


def with_ancestors(direct, names, parents):
    """Add every parent term, so a specific role implies the general one.

    A gene annotated to "cellular response to glucocorticoid stimulus" is
    also responding to a hormone and to a chemical, and a test that does
    not know this finds nothing at the levels people think in.
    """
    reachable, full = {}, {}

    def above(term):
        if term not in reachable:
            found, stack = set(), [term]
            while stack:
                for parent in parents.get(stack.pop(), ()):
                    if parent not in found:
                        found.add(parent)
                        stack.append(parent)
            reachable[term] = found
        return reachable[term]

    for symbol, terms in direct.items():
        every = set()
        for term in terms:
            if term in names:
                every |= {term} | above(term)
        full[symbol] = {term for term in every
                        if names[term][1] == PROCESS}
    return full


def benjamini_hochberg(pvalues):
    """The step-up correction of Chapter 19, written out."""
    order = sorted(range(len(pvalues)), key=lambda i: pvalues[i])
    adjusted, smallest = [1.0] * len(pvalues), 1.0
    for rank, index in enumerate(reversed(order), 1):
        position = len(pvalues) - rank + 1
        smallest = min(smallest, pvalues[index] * len(pvalues) / position)
        adjusted[index] = smallest
    return adjusted


def test(sets, genes, background, names, direction):
    rows = []
    for term, members in sets.items():
        found = members & genes
        if len(found) < MIN_FOUND:
            continue
        rows.append({
            "term": term,
            "name": names[term][0],
            "direction": direction,
            "found": len(found),
            "in_set": len(members),
            "expected": round(
                len(members) * len(genes) / len(background), 1),
            "p": hypergeom.sf(len(found) - 1, len(background),
                              len(members), len(genes)),
            "genes": " ".join(sorted(found)[:8]),
        })
    for row, q in zip(rows, benjamini_hochberg([r["p"] for r in rows])):
        row["q"] = q
    return rows


def main():
    names, parents = ontology()
    full = with_ancestors(annotations(), names, parents)
    results = pd.read_csv("results.tsv", sep="\t").dropna(
        subset=["padj", "symbol"])

    background = {s for s in results["symbol"] if s in full}
    changed = results[(results["padj"] < 0.05)
                      & (results["log2FoldChange"].abs() >= 1)]
    sets = {}
    for symbol in background:
        for term in full[symbol]:
            sets.setdefault(term, set()).add(symbol)
    sets = {term: members for term, members in sets.items()
            if SMALLEST <= len(members) <= LARGEST}

    rows = []
    for direction, wanted in (("up", changed["log2FoldChange"] > 0),
                              ("down", changed["log2FoldChange"] < 0)):
        genes = {s for s in changed[wanted]["symbol"] if s in full}
        rows += test(sets, genes, background, names, direction)
        print(f"{len(genes)} genes {direction}")
    rows.sort(key=lambda row: (row["direction"], row["p"]))
    with open("enrichment.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=rows[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

    print(f"{len(background):,} genes carry an annotation, and "
          f"{len(sets):,} terms are testable")
    for direction in ("up", "down"):
        found = [row for row in rows if row["direction"] == direction
                 and row["q"] < 0.05]
        print(f"\n{len(found)} terms enriched among the genes that went "
              f"{direction}:")
        for row in found[:8]:
            print(f"  q={row['q']:.0e} {row['found']:3} of "
                  f"{row['in_set']:3}  {row['name'][:42]}")


if __name__ == "__main__":
    main()

## `compare.py`

Compare the findings with the paper the data came from.


In [ ]:
"""Compare the findings with the paper the data came from.

GSE52778 was published by Himes and others in 2014. They reported 316
differentially expressed genes at a Benjamini-Hochberg corrected p below
0.05, named four well known glucocorticoid responders and three less
studied ones, and followed CRISPLD2 into the laboratory. A reanalysis
that cannot find their genes has gone wrong somewhere.

uv run compare.py           writes comparison.tsv
"""

import csv

import pandas as pd

PAPER = {
    "DUSP1": "well known", "KLF15": "well known", "PER1": "well known",
    "TSC22D3": "well known", "C7": "less investigated",
    "CCDC69": "less investigated", "CRISPLD2": "less investigated",
}
THEIR_COUNT = 316


def main():
    results = pd.read_csv("results.tsv", sep="\t").dropna(subset=["padj"])
    results = results.sort_values("padj").reset_index(drop=True)
    results["rank"] = results.index + 1

    rows = []
    for symbol, standing in PAPER.items():
        found = results[results["symbol"] == symbol]
        if found.empty:
            rows.append({"symbol": symbol, "in_paper": standing,
                         "rank": "", "log2FoldChange": "", "padj": ""})
            continue
        gene = found.iloc[0]
        rows.append({
            "symbol": symbol,
            "in_paper": standing,
            "rank": int(gene["rank"]),
            "log2FoldChange": round(gene["log2FoldChange"], 2),
            "padj": f"{gene['padj']:.1e}",
        })
    with open("comparison.tsv", "w", newline="") as out:
        writer = csv.DictWriter(out, fieldnames=rows[0], delimiter="\t")
        writer.writeheader()
        writer.writerows(rows)

    print("the seven genes the paper names, in this reanalysis:\n")
    print(f"{'gene':10}{'in the paper':20}{'rank':>6}{'log2FC':>9}"
          f"{'padj':>10}")
    for row in rows:
        print(f"{row['symbol']:10}{row['in_paper']:20}{row['rank']:>6}"
              f"{row['log2FoldChange']:>9}{row['padj']:>10}")

    ours = (results["padj"] < 0.05).sum()
    print(f"\nthey reported {THEIR_COUNT} changed genes, this analysis "
          f"{ours:,}")
    print(f"all {len(rows)} of their named genes are in the top "
          f"{max(int(r['rank']) for r in rows if r['rank'])} here, "
          "and all went up")


if __name__ == "__main__":
    main()